# Review embeddings (D20)

Checks that every non-short review has a vector, and that nearest neighbours share a topic rather than just a game.

In [ ]:
import os, glob, warnings, numpy as np, pandas as pd, psycopg, pyarrow as pa, pyarrow.parquet as pq
from dotenv import load_dotenv
warnings.filterwarnings("ignore", message="pandas only supports SQLAlchemy")
load_dotenv("../.env")
conn = psycopg.connect(os.environ["DATABASE_URL"], autocommit=True, connect_timeout=10)

In [ ]:
parts = sorted(glob.glob("../data/embeddings/part-*.parquet"))
done = sum(pq.read_metadata(f).num_rows for f in parts)      # reads only each file's footer
todo = pd.read_sql("SELECT count(*) FROM reviews WHERE NOT is_short", conn).iat[0, 0]
print(f"{len(parts)} part files, {done:,} embedded of {todo:,} non-short reviews")

In [ ]:
rng = np.random.default_rng(20261023)
pick = rng.choice(parts, 5, replace=False)              # part files are in ID order, so spread the sample out
t = pa.concat_tables([pq.read_table(p) for p in pick])
ids = t.column("recommendationid").to_numpy()
vecs = t.column("embedding").combine_chunks().flatten().to_numpy().reshape(-1, 384)
norms = np.linalg.norm(vecs, axis=1)
print(f"{len(ids):,} reviews; norms min {norms.min():.4f}, max {norms.max():.4f}")   # both ~1.0000

meta = (pd.read_sql("""SELECT r.recommendationid, r.appid, g.name, r.review
                          FROM reviews r JOIN games g USING (appid)
                          WHERE r.recommendationid = ANY(%(ids)s)""",
                       conn, params={"ids": ids.tolist()})
             .set_index("recommendationid").loc[ids])

In [ ]:
def neighbours(word, k=10, pick = 0):
    cands = [i for i, r in enumerate(meta.review) if word in r.lower() and 40 < len(r) < 300]
    q = cands[pick]
    sims = vecs @ vecs[q]                               # unit vectors, so dot product = cosine similarity
    top = np.argsort(-sims)[1:k + 1]                    # [0] is the query itself
    same = (meta.appid.iloc[top] == meta.appid.iloc[q]).to_numpy()
    print(f"QUERY '{word}' ({len(cands)} matches): {meta.review.iloc[q][:200]!r}\n")
    for j, s in zip(top, same):
        print(f"{sims[j]:.2f}  {'same game' if s else '         '}  {meta.review.iloc[j][:150]!r}")
    base = vecs[rng.integers(0, len(vecs), 1000)] @ vecs[q]
    print(f"\ntop {k}: {sims[top].min():.2f}–{sims[top].max():.2f}; random reviews: {base.mean():.2f}; "
          f"{same.sum()} of {k} from the query's game")

In [ ]:
def full_top(word, k=10, pick = 0):
    """Full text and game of the query and its top k neighbours (same picks as neighbours())."""
    cands = [i for i, r in enumerate(meta.review) if word in r.lower() and 40 < len(r) < 300]
    q = cands[pick]
    sims = vecs @ vecs[q]
    top = np.argsort(-sims)[1:k + 1]
    print(f"QUERY '{word}' — {meta.name.iloc[q]} (appid {meta.appid.iloc[q]})")
    print(meta.review.iloc[q].strip())
    print("=" * 100)
    for rank, j in enumerate(top, 1):
        tag = "  << SAME GAME" if meta.appid.iloc[j] == meta.appid.iloc[q] else ""
        print(f"#{rank}  similarity {sims[j]:.2f}  |  {meta.name.iloc[j]} (appid {meta.appid.iloc[j]}){tag}")
        print(meta.review.iloc[j].strip())
        print("-" * 100)

In [ ]:
def candidates(word, n=10):
    cands = [i for i, r in enumerate(meta.review) if word in r.lower() and 40 < len(r) < 300]
    for k, i in enumerate(cands[:n]):
        print(f"pick={k}  {meta.name.iloc[i][:25]:25}  {meta.review.iloc[i][:200]!r}")

In [ ]:
candidates("crash")
full_top("crash", pick=0)
# 8/10 of the top 10 are about crashes or bugs

In [ ]:
candidates("refund")
full_top("refund", pick=0)

# 4 reviews from 2 games
# most messages are about co-op, with "Refunded" as an aside, since the embeds follow the main complaint

In [ ]:
candidates("matchmaking")
full_top("matchmaking", pick=1)

# 2 of the top 10 are from 1 game
# 10 of the top 10 are about matchmaking

8 of the top 10 are about crashes or freezes.